# Music Generation with Neural Networks on JSB Chorales

This notebook explores two symbolic music generation tasks using Bach's chorales:

1. **Unconditioned generation** — train a language model to learn the distribution of four-part chorale sequences and sample new music from it
2. **Conditioned generation (harmonization)** — given a soprano melody, generate the alto, tenor, and bass voices

We use the JSB Chorales dataset (Boulanger-Lewandowski et al., 2012), which contains 382 Bach chorales quantized at 16th-note resolution, each represented as sequences of four MIDI pitches (soprano, alto, tenor, bass).

## 0. Setup

In [ ]:
!pip install -q torch music21 pretty_midi matplotlib seaborn numpy pandas

In [ ]:
import os
import pickle
import urllib.request
import random
import math
import copy
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from torch.optim.lr_scheduler import CosineAnnealingLR

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import pretty_midi
from IPython.display import display, Audio

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Data Pipeline

The JSB Chorales dataset contains 382 four-part Bach chorales at 16th-note resolution. Each timestep is a chord of four MIDI pitches representing soprano, alto, tenor, and bass voices. The dataset comes pre-split into training (229), validation (76), and test (77) sets.

We download the standard pickle format from the Boulanger-Lewandowski repository, which is the canonical version used in most music generation research.

In [ ]:
DATA_URL = 'https://raw.githubusercontent.com/czhuang/JSB-Chorales-dataset/master/jsb-chorales-16th.pkl'
DATA_PATH = 'jsb-chorales-16th.pkl'

if not os.path.exists(DATA_PATH):
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)
    print('Downloaded JSB Chorales dataset')

with open(DATA_PATH, 'rb') as f:
    raw_data = pickle.load(f, encoding='latin1')

print(f'Raw splits — Train: {len(raw_data["train"])}, Valid: {len(raw_data["valid"])}, Test: {len(raw_data["test"])}')

### Preprocessing

Each chorale is a list of chords, where each chord is a set of active MIDI pitches. Most chords have exactly 4 notes (one per voice), but a small fraction (~0.4%) have fewer — these represent rests or incomplete voicings. We filter these out within each sequence while keeping the sequence itself intact.

We assign voices by register: the highest pitch becomes soprano, next highest alto, then tenor, then bass. This is the standard approach for this dataset since the original encoding stores pitches as unordered sets.

In [ ]:
def parse_chorales(split_data):
    """Parse raw pickle data into lists of (S, A, T, B) tuples, filtering non-4-note chords."""
    chorales = []
    total_chords = 0
    filtered_chords = 0
    for seq in split_data:
        clean = []
        for chord in seq:
            total_chords += 1
            if len(chord) == 4:
                clean.append(tuple(sorted(chord, reverse=True)))
            else:
                filtered_chords += 1
        if len(clean) >= 10:
            chorales.append(clean)
    return chorales, total_chords, filtered_chords

train_chorales, train_total, train_filtered = parse_chorales(raw_data['train'])
valid_chorales, valid_total, valid_filtered = parse_chorales(raw_data['valid'])
test_chorales, test_total, test_filtered = parse_chorales(raw_data['test'])

total_filtered = train_filtered + valid_filtered + test_filtered
total_chords = train_total + valid_total + test_total

print(f'After cleaning — Train: {len(train_chorales)}, Valid: {len(valid_chorales)}, Test: {len(test_chorales)}')
print(f'Filtered {total_filtered}/{total_chords} non-4-note chords ({100*total_filtered/total_chords:.1f}%)')

### Vocabulary

We build a shared vocabulary across all four voices — since they operate in the same MIDI pitch space, sharing embeddings lets the model learn that a given pitch means the same thing regardless of which voice plays it. We reserve three special tokens for padding, sequence boundaries.

In [ ]:
PAD_TOKEN = 0
BOS_TOKEN = 1
EOS_TOKEN = 2

all_pitches = set()
for seq in train_chorales:
    for chord in seq:
        all_pitches.update(chord)

sorted_pitches = sorted(all_pitches)
pitch_to_idx = {p: i + 3 for i, p in enumerate(sorted_pitches)}
idx_to_pitch = {v: k for k, v in pitch_to_idx.items()}
idx_to_pitch[PAD_TOKEN] = None
idx_to_pitch[BOS_TOKEN] = None
idx_to_pitch[EOS_TOKEN] = None

VOCAB_SIZE = len(pitch_to_idx) + 3
print(f'Vocabulary size: {VOCAB_SIZE} ({len(sorted_pitches)} pitches + 3 special tokens)')
print(f'Pitch range: MIDI {sorted_pitches[0]} ({pretty_midi.note_number_to_name(sorted_pitches[0])}) to {sorted_pitches[-1]} ({pretty_midi.note_number_to_name(sorted_pitches[-1])})')

In [ ]:
def encode_chord(chord):
    """Convert (S, A, T, B) MIDI pitches to token indices."""
    encoded = []
    for p in chord:
        if p in pitch_to_idx:
            encoded.append(pitch_to_idx[p])
        else:
            nearest = min(pitch_to_idx.keys(), key=lambda k: abs(k - p))
            encoded.append(pitch_to_idx[nearest])
    return encoded

def decode_token(token):
    """Convert a single token index back to a MIDI pitch."""
    return idx_to_pitch.get(token)

### Data Augmentation

With only 229 training chorales, overfitting is a real concern. We augment the training data by transposing each chorale by a few semitone shifts. This is musically valid — transposition preserves harmonic structure — and expands our training set by 4x. We only augment the training split to avoid data leakage.

In [ ]:
def augment_chorales(chorales, shifts=range(-5, 7)):
    """Transpose chorales by the given semitone shifts."""
    augmented = []
    for seq in chorales:
        for shift in shifts:
            transposed = [tuple(p + shift for p in chord) for chord in seq]
            augmented.append(transposed)
    return augmented

train_chorales_aug = augment_chorales(train_chorales, shifts=[-3, 0, 3, 6])

# Rebuild vocab to include augmented pitches
all_pitches_aug = set()
for seq in train_chorales_aug:
    for chord in seq:
        all_pitches_aug.update(chord)

sorted_pitches = sorted(all_pitches_aug)
pitch_to_idx = {p: i + 3 for i, p in enumerate(sorted_pitches)}
idx_to_pitch = {v: k for k, v in pitch_to_idx.items()}
idx_to_pitch[PAD_TOKEN] = None
idx_to_pitch[BOS_TOKEN] = None
idx_to_pitch[EOS_TOKEN] = None

VOCAB_SIZE = len(pitch_to_idx) + 3
print(f'Augmented training set: {len(train_chorales_aug)} chorales (from {len(train_chorales)})')
print(f'Updated vocabulary size: {VOCAB_SIZE} ({len(sorted_pitches)} pitches + 3 special tokens)')
print(f'Augmented pitch range: MIDI {sorted_pitches[0]} to {sorted_pitches[-1]}')

### Exploratory Data Analysis

Let's examine the structure of the dataset before modeling.

In [ ]:
# Pitch distribution per voice (using original, non-augmented data for interpretability)
fig, axes = plt.subplots(1, 4, figsize=(20, 4), sharey=True)
voice_names = ['Soprano', 'Alto', 'Tenor', 'Bass']
colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']

for v, (ax, name, color) in enumerate(zip(axes, voice_names, colors)):
    pitches = [chord[v] for seq in train_chorales for chord in seq]
    ax.hist(pitches, bins=range(35, 85), color=color, alpha=0.8, edgecolor='black', linewidth=0.3)
    ax.set_title(f'{name} (range: {min(pitches)}-{max(pitches)})')
    ax.set_xlabel('MIDI Pitch')

axes[0].set_ylabel('Count')
plt.suptitle('Pitch Distribution per Voice (Training Set)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Sequence length histogram
all_lengths = [len(seq) for seq in train_chorales + valid_chorales + test_chorales]

plt.figure(figsize=(10, 4))
plt.hist(all_lengths, bins=40, edgecolor='black', alpha=0.7, color='steelblue')
plt.axvline(np.mean(all_lengths), color='red', linestyle='--', label=f'Mean: {np.mean(all_lengths):.0f} steps')
plt.xlabel('Sequence Length (16th-note timesteps)')
plt.ylabel('Count')
plt.title('Chorale Length Distribution')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Soprano-Bass co-occurrence heatmap
soprano_pitches = [chord[0] for seq in train_chorales for chord in seq]
bass_pitches = [chord[3] for seq in train_chorales for chord in seq]

co_occ = pd.crosstab(
    pd.Series(soprano_pitches, name='Soprano MIDI'),
    pd.Series(bass_pitches, name='Bass MIDI')
)

plt.figure(figsize=(12, 8))
sns.heatmap(co_occ, cmap='YlOrRd', linewidths=0.1)
plt.title('Soprano–Bass Pitch Co-occurrence (Training Set)')
plt.tight_layout()
plt.show()

In [ ]:
# Pitch-class distribution (mod 12) — reveals key tendencies
pitch_class_names = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
all_train_pitches = [p for seq in train_chorales for chord in seq for p in chord]
pitch_classes = [p % 12 for p in all_train_pitches]

pc_counts = Counter(pitch_classes)
pc_values = [pc_counts.get(i, 0) for i in range(12)]

plt.figure(figsize=(10, 4))
plt.bar(range(12), pc_values, tick_label=pitch_class_names, color='teal', alpha=0.8, edgecolor='black')
plt.xlabel('Pitch Class')
plt.ylabel('Count')
plt.title('Pitch-Class Distribution (Training Set) — Key Structure Tendencies')
plt.tight_layout()
plt.show()

In [ ]:
# Unisons: cases where adjacent voices share the same pitch
unisons = 0
total_pairs = 0
for seq in train_chorales:
    for chord in seq:
        for i in range(3):
            total_pairs += 1
            if chord[i] == chord[i + 1]:
                unisons += 1

# Summary statistics
stats = pd.DataFrame({
    'Metric': [
        'Vocabulary size',
        'Training sequences (original)',
        'Training sequences (augmented)',
        'Validation sequences',
        'Test sequences',
        'Avg sequence length',
        'Min / Max length',
        'Unisons (adjacent voices on same pitch)',
    ],
    'Value': [
        VOCAB_SIZE,
        len(train_chorales),
        len(train_chorales_aug),
        len(valid_chorales),
        len(test_chorales),
        f'{np.mean([len(s) for s in train_chorales]):.0f} steps',
        f'{min(len(s) for s in train_chorales)} / {max(len(s) for s in train_chorales)}',
        f'{unisons}/{total_pairs} ({100*unisons/total_pairs:.1f}%)',
    ]
})
display(stats)

The pitch distributions show clear voice ranges with some overlap — soprano tends to sit in the 60-80 MIDI range, bass in 36-65. The pitch-class histogram reveals a bias toward natural notes (C, D, E, F, G, A, B), consistent with Bach's predominantly diatonic writing. Unisons (adjacent voices on the same pitch) occur in a small percentage of chords, which is typical in four-part writing where voices occasionally converge.

---

## 2. Task 1 — Symbolic Unconditioned Generation

**Goal:** Learn the distribution $p(x)$ over four-part chorale sequences and sample new music from it.

We frame this as a language modeling problem. Each chorale is flattened into an interleaved sequence of tokens: $[S_1, A_1, T_1, B_1, S_2, A_2, T_2, B_2, \ldots]$, where each token is a MIDI pitch index. The model learns to predict the next token given all previous tokens: $p(x_t \mid x_{<t})$.

This interleaving captures vertical (harmonic) relationships through the S→A→T→B pattern within each chord, and horizontal (melodic) relationships across consecutive chords.

### Baseline: Bigram Markov Chain

As a baseline, we use a first-order Markov chain that models $p(x_t \mid x_{t-1})$ — the probability of each token given only the immediately preceding token.

In [ ]:
def chorales_to_interleaved(chorales):
    """Flatten chorales into interleaved SATB token sequences."""
    sequences = []
    for seq in chorales:
        tokens = [BOS_TOKEN]
        for chord in seq:
            tokens.extend(encode_chord(chord))
        tokens.append(EOS_TOKEN)
        sequences.append(tokens)
    return sequences

train_seqs = chorales_to_interleaved(train_chorales_aug)
valid_seqs = chorales_to_interleaved(valid_chorales)
test_seqs = chorales_to_interleaved(test_chorales)

print(f'Interleaved sequence lengths — mean: {np.mean([len(s) for s in train_seqs]):.0f}, '
      f'min: {min(len(s) for s in train_seqs)}, max: {max(len(s) for s in train_seqs)}')

In [ ]:
class BigramMarkov:
    def __init__(self):
        self.transitions = defaultdict(Counter)

    def fit(self, sequences):
        for seq in sequences:
            for i in range(len(seq) - 1):
                self.transitions[seq[i]][seq[i + 1]] += 1
        # Precompute normalized probabilities
        self.probs = {}
        for prev, counts in self.transitions.items():
            total = sum(counts.values())
            self.probs[prev] = {tok: c / total for tok, c in counts.items()}

    def perplexity(self, sequences):
        total_logp = 0.0
        total_tokens = 0
        for seq in sequences:
            for i in range(len(seq) - 1):
                prev, nxt = seq[i], seq[i + 1]
                prob = self.probs.get(prev, {}).get(nxt, 1e-6)
                total_logp += math.log(prob)
                total_tokens += 1
        return math.exp(-total_logp / total_tokens)

    def sample(self, length=400, temperature=1.0):
        tokens = [BOS_TOKEN]
        for _ in range(length):
            prev = tokens[-1]
            if prev not in self.probs:
                break
            candidates = list(self.probs[prev].keys())
            weights = np.array([self.probs[prev][c] for c in candidates])
            weights = weights ** (1.0 / temperature)
            weights /= weights.sum()
            nxt = np.random.choice(candidates, p=weights)
            if nxt == EOS_TOKEN:
                break
            tokens.append(nxt)
        return tokens[1:]

markov = BigramMarkov()
markov.fit(train_seqs)
markov_test_ppl = markov.perplexity(test_seqs)
print(f'Bigram Markov — test perplexity: {markov_test_ppl:.2f}')

### LSTM Language Model

Our main model is a 2-layer LSTM that processes the interleaved token sequence autoregressively. The architecture is: embedding (64-dim) → LSTM (128 hidden, 2 layers) → linear projection to vocabulary.

We chose this size to avoid overfitting — even with augmentation, the dataset is relatively small, and the vocabulary is only ~60 tokens. Dropout is applied between LSTM layers and on the embedding output.

In [ ]:
CHUNK_LEN = 256

class ChunkedLMDataset(Dataset):
    """Splits sequences into fixed-length chunks for language model training."""
    def __init__(self, sequences, chunk_len=CHUNK_LEN):
        self.chunks = []
        for seq in sequences:
            t = torch.tensor(seq, dtype=torch.long)
            for i in range(0, len(t) - 1, chunk_len):
                inp = t[i:i + chunk_len]
                tgt = t[i + 1:i + chunk_len + 1]
                if len(inp) == len(tgt) and len(inp) > 0:
                    self.chunks.append((inp, tgt))

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        return self.chunks[idx]

def lm_collate(batch):
    inputs, targets = zip(*batch)
    return (pad_sequence(inputs, batch_first=True, padding_value=PAD_TOKEN),
            pad_sequence(targets, batch_first=True, padding_value=PAD_TOKEN))

BATCH_SIZE = 32
train_lm = DataLoader(ChunkedLMDataset(train_seqs), batch_size=BATCH_SIZE, shuffle=True, collate_fn=lm_collate)
valid_lm = DataLoader(ChunkedLMDataset(valid_seqs), batch_size=BATCH_SIZE, shuffle=False, collate_fn=lm_collate)
test_lm = DataLoader(ChunkedLMDataset(test_seqs), batch_size=BATCH_SIZE, shuffle=False, collate_fn=lm_collate)

print(f'Training chunks: {len(train_lm.dataset)}, Valid: {len(valid_lm.dataset)}, Test: {len(test_lm.dataset)}')

In [ ]:
class LSTMLanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_TOKEN)
        self.embed_dropout = nn.Dropout(0.2)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                           dropout=dropout, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

    def forward(self, x, hidden=None):
        emb = self.embed_dropout(self.embedding(x))
        out, hidden = self.lstm(emb, hidden)
        return self.fc(out), hidden

    def init_hidden(self, batch_size, device):
        return (torch.zeros(self.num_layers, batch_size, self.hidden_dim, device=device),
                torch.zeros(self.num_layers, batch_size, self.hidden_dim, device=device))

lm_model = LSTMLanguageModel(VOCAB_SIZE).to(device)
total_params = sum(p.numel() for p in lm_model.parameters() if p.requires_grad)
print(f'LSTM LM parameters: {total_params:,}')

In [ ]:
def train_lm_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    total_tokens = 0
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        logits, _ = model(inputs)
        loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        mask = targets != PAD_TOKEN
        total_loss += loss.item()
        total_tokens += mask.sum().item()
    return math.exp(total_loss / total_tokens)

@torch.no_grad()
def eval_lm(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        logits, _ = model(inputs)
        loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        mask = targets != PAD_TOKEN
        total_loss += loss.item()
        total_tokens += mask.sum().item()
    return math.exp(total_loss / total_tokens)

In [ ]:
NUM_EPOCHS = 30
criterion = nn.CrossEntropyLoss(ignore_index=PAD_TOKEN, reduction='sum')
optimizer = torch.optim.Adam(lm_model.parameters(), lr=1e-3)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-5)

train_ppls, valid_ppls = [], []
best_valid_ppl = float('inf')

for epoch in range(1, NUM_EPOCHS + 1):
    train_ppl = train_lm_epoch(lm_model, train_lm, optimizer, criterion)
    valid_ppl = eval_lm(lm_model, valid_lm, criterion)
    scheduler.step()
    train_ppls.append(train_ppl)
    valid_ppls.append(valid_ppl)

    if valid_ppl < best_valid_ppl:
        best_valid_ppl = valid_ppl
        torch.save(lm_model.state_dict(), 'best_lm.pt')

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d} | Train PPL: {train_ppl:.2f} | Valid PPL: {valid_ppl:.2f} | LR: {scheduler.get_last_lr()[0]:.6f}')

lm_model.load_state_dict(torch.load('best_lm.pt', weights_only=True))
print(f'\nBest validation perplexity: {best_valid_ppl:.2f}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, NUM_EPOCHS + 1), train_ppls, label='Train', linewidth=1.5)
ax.plot(range(1, NUM_EPOCHS + 1), valid_ppls, label='Validation', linewidth=1.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('Perplexity')
ax.set_title('Task 1: LSTM Language Model — Learning Curves')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Evaluation

In [ ]:
lstm_test_ppl = eval_lm(lm_model, test_lm, criterion)

print(f'Test Perplexity Comparison:')
print(f'  Bigram Markov:  {markov_test_ppl:.2f}')
print(f'  LSTM:           {lstm_test_ppl:.2f}')
print(f'  Random baseline: {VOCAB_SIZE:.0f} (uniform over vocab)')

In [ ]:
@torch.no_grad()
def sample_lstm(model, max_len=400, temperature=1.0):
    model.eval()
    tokens = [BOS_TOKEN]
    hidden = model.init_hidden(1, device)
    for _ in range(max_len):
        x = torch.tensor([[tokens[-1]]], device=device)
        logits, hidden = model(x, hidden)
        logits = logits[0, -1] / temperature
        probs = F.softmax(logits, dim=-1)
        probs[PAD_TOKEN] = 0
        probs = probs / probs.sum()
        nxt = torch.multinomial(probs, 1).item()
        if nxt == EOS_TOKEN:
            break
        tokens.append(nxt)
    return tokens[1:]

def tokens_to_chords(tokens):
    """Convert interleaved tokens back to (S, A, T, B) chord tuples."""
    chords = []
    for i in range(0, len(tokens) - 3, 4):
        pitches = [decode_token(tokens[i + v]) for v in range(4)]
        if all(p is not None and isinstance(p, int) for p in pitches):
            chords.append(tuple(pitches))
    return chords

In [ ]:
# Sample at different temperatures
temperatures = [0.5, 1.0, 1.5]
generated = {}

for temp in temperatures:
    samples = [tokens_to_chords(sample_lstm(lm_model, temperature=temp)) for _ in range(4)]
    generated[temp] = samples
    lengths = [len(s) for s in samples]
    print(f'T={temp}: generated {len(samples)} samples, avg {np.mean(lengths):.0f} chords')

In [ ]:
# Pitch distribution: generated vs real
real_pitches = [p for seq in train_chorales for chord in seq for p in chord]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, temp in zip(axes, temperatures):
    gen_pitches = [p for s in generated[temp] for chord in s for p in chord]
    bins = range(30, 90)
    ax.hist(real_pitches, bins=bins, alpha=0.5, density=True, label='Real', color='steelblue')
    if gen_pitches:
        ax.hist(gen_pitches, bins=bins, alpha=0.5, density=True, label=f'Generated', color='coral')
    ax.set_title(f'Temperature = {temp}')
    ax.set_xlabel('MIDI Pitch')
    ax.legend()

axes[0].set_ylabel('Density')
plt.suptitle('Pitch Distribution: Real vs Generated', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Pitch-class distribution comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
real_pc = [p % 12 for p in real_pitches]
real_pc_counts = Counter(real_pc)
real_pc_norm = np.array([real_pc_counts.get(i, 0) for i in range(12)], dtype=float)
real_pc_norm /= real_pc_norm.sum()

for ax, temp in zip(axes, temperatures):
    gen_pitches = [p for s in generated[temp] for chord in s for p in chord]
    gen_pc = [p % 12 for p in gen_pitches]
    gen_pc_counts = Counter(gen_pc)
    gen_pc_norm = np.array([gen_pc_counts.get(i, 0) for i in range(12)], dtype=float)
    if gen_pc_norm.sum() > 0:
        gen_pc_norm /= gen_pc_norm.sum()

    x = np.arange(12)
    ax.bar(x - 0.2, real_pc_norm, 0.4, label='Real', color='steelblue', alpha=0.8)
    ax.bar(x + 0.2, gen_pc_norm, 0.4, label='Generated', color='coral', alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(pitch_class_names)
    ax.set_title(f'Temperature = {temp}')
    ax.legend()

axes[0].set_ylabel('Proportion')
plt.suptitle('Pitch-Class Distribution: Real vs Generated', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Voice range violations
VOICE_RANGES = {
    'Soprano': (60, 81), 'Alto': (52, 74),
    'Tenor': (46, 69), 'Bass': (36, 66)
}

def count_range_violations(chords):
    violations = 0
    total = 0
    for chord in chords:
        for v, (name, (lo, hi)) in enumerate(VOICE_RANGES.items()):
            if v < len(chord):
                total += 1
                if chord[v] < lo or chord[v] > hi:
                    violations += 1
    return violations, total

print('Voice range violations (notes outside standard ranges):')
for temp in temperatures:
    v, t = 0, 0
    for s in generated[temp]:
        vi, ti = count_range_violations(s)
        v += vi; t += ti
    print(f'  T={temp}: {v}/{t} ({100*v/max(t,1):.1f}%)')

rv, rt = 0, 0
for seq in test_chorales:
    vi, ti = count_range_violations(seq)
    rv += vi; rt += ti
print(f'  Real (test): {rv}/{rt} ({100*rv/max(rt,1):.1f}%)')

The LSTM substantially outperforms the Markov baseline in perplexity, demonstrating that it captures longer-range dependencies in the chorale structure. Lower temperatures produce more conservative (and more realistic) outputs, while higher temperatures introduce more variety at the cost of occasionally violating voice ranges or producing unusual intervals.

The pitch-class distributions show that the model learns the key structure of the training data — the bias toward natural notes (C, D, E, G, A) is preserved in generation, especially at lower temperatures.

### MIDI Export

In [ ]:
def chorales_to_midi(chords, filename, tempo=100):
    """Export (S, A, T, B) chords to a MIDI file, merging consecutive identical pitches."""
    pm = pretty_midi.PrettyMIDI(initial_tempo=tempo)
    step_dur = 60.0 / tempo / 4  # 16th note duration

    for v in range(4):
        inst = pretty_midi.Instrument(program=0, name=['Soprano', 'Alto', 'Tenor', 'Bass'][v])
        i = 0
        while i < len(chords):
            pitch = chords[i][v]
            start = i * step_dur
            j = i + 1
            while j < len(chords) and chords[j][v] == pitch:
                j += 1
            end = j * step_dur
            inst.notes.append(pretty_midi.Note(velocity=80, pitch=int(pitch), start=start, end=end))
            i = j
        pm.instruments.append(inst)

    pm.write(filename)
    return filename

# Export best sample (T=1.0)
best_sample = max(generated[1.0], key=len)
chorales_to_midi(best_sample, 'symbolic_unconditioned.mid')
print(f'Exported symbolic_unconditioned.mid ({len(best_sample)} chords)')

---

## 3. Task 2 — Symbolic Conditioned Generation (Harmonization)

**Goal:** Given a soprano melody, generate musically coherent alto, tenor, and bass voices.

This is a sequence labeling problem, not a translation problem — the soprano and accompaniment are timestep-aligned, so there's no length mismatch or reordering to handle. A bidirectional LSTM is ideal here: it reads the full soprano melody in both directions, giving each timestep context about where the melody came from *and* where it's going. This context is critical for harmonization — a V-I cadence requires knowing that the melody is about to resolve.

Three parallel output heads predict the alto, tenor, and bass pitches simultaneously at each timestep.

### Baseline: Frequency-Based Harmonization

For each soprano pitch, predict the most frequently observed (alto, tenor, bass) combination in the training data. This captures the most common harmonization choices but ignores sequential context entirely.

In [ ]:
class HarmonizationDataset(Dataset):
    def __init__(self, chorales):
        self.soprano = []
        self.targets = []
        for seq in chorales:
            sop = torch.tensor([encode_chord(c)[0] for c in seq], dtype=torch.long)
            alto = torch.tensor([encode_chord(c)[1] for c in seq], dtype=torch.long)
            tenor = torch.tensor([encode_chord(c)[2] for c in seq], dtype=torch.long)
            bass = torch.tensor([encode_chord(c)[3] for c in seq], dtype=torch.long)
            self.soprano.append(sop)
            self.targets.append(torch.stack([alto, tenor, bass], dim=-1))

    def __len__(self):
        return len(self.soprano)

    def __getitem__(self, idx):
        return self.soprano[idx], self.targets[idx]

def harm_collate(batch):
    sops, tgts = zip(*batch)
    lengths = torch.tensor([len(s) for s in sops])
    sops_pad = pad_sequence(sops, batch_first=True, padding_value=PAD_TOKEN)
    tgts_pad = pad_sequence(tgts, batch_first=True, padding_value=PAD_TOKEN)
    return sops_pad, tgts_pad, lengths

train_harm = DataLoader(HarmonizationDataset(train_chorales_aug), batch_size=16, shuffle=True, collate_fn=harm_collate)
valid_harm = DataLoader(HarmonizationDataset(valid_chorales), batch_size=16, shuffle=False, collate_fn=harm_collate)
test_harm = DataLoader(HarmonizationDataset(test_chorales), batch_size=16, shuffle=False, collate_fn=harm_collate)

print(f'Harmonization batches — Train: {len(train_harm)}, Valid: {len(valid_harm)}, Test: {len(test_harm)}')

In [ ]:
class FrequencyBaseline:
    def __init__(self):
        self.chord_map = defaultdict(Counter)

    def fit(self, chorales):
        for seq in chorales:
            for s, a, t, b in seq:
                self.chord_map[s][(a, t, b)] += 1

    def predict(self, soprano_seq):
        result = []
        for s in soprano_seq:
            if s in self.chord_map:
                atb = self.chord_map[s].most_common(1)[0][0]
            else:
                nearest = min(self.chord_map.keys(), key=lambda k: abs(k - s))
                atb = self.chord_map[nearest].most_common(1)[0][0]
            result.append(atb)
        return result

    def perplexity(self, chorales):
        total_logp = 0.0
        total_tokens = 0
        for seq in chorales:
            for s, a, t, b in seq:
                total = sum(self.chord_map[s].values()) if s in self.chord_map else 0
                count = self.chord_map.get(s, {}).get((a, t, b), 0)
                prob = (count + 1) / (total + len(self.chord_map.get(s, {})) + 1)
                total_logp += math.log(prob)
                total_tokens += 1
        return math.exp(-total_logp / total_tokens)

freq_baseline = FrequencyBaseline()
freq_baseline.fit(train_chorales)
freq_test_ppl = freq_baseline.perplexity(test_chorales)
print(f'Frequency baseline — test perplexity: {freq_test_ppl:.2f}')

### Bidirectional LSTM Harmonizer

The model reads the full soprano sequence bidirectionally, then predicts alto, tenor, and bass at each timestep through three independent linear heads. The bidirectional context is crucial — harmonization depends on knowing what comes next in the melody (e.g., approaching a cadence requires preparing the harmonic resolution).

In [ ]:
class BiLSTMHarmonizer(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_TOKEN)
        self.embed_dropout = nn.Dropout(0.2)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                           dropout=dropout, batch_first=True, bidirectional=True)
        self.head_alto = nn.Linear(hidden_dim * 2, vocab_size)
        self.head_tenor = nn.Linear(hidden_dim * 2, vocab_size)
        self.head_bass = nn.Linear(hidden_dim * 2, vocab_size)

    def forward(self, src, lengths):
        emb = self.embed_dropout(self.embedding(src))
        packed = pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, _ = pad_packed_sequence(out, batch_first=True)
        return self.head_alto(out), self.head_tenor(out), self.head_bass(out)

harm_model = BiLSTMHarmonizer(VOCAB_SIZE).to(device)
total_params = sum(p.numel() for p in harm_model.parameters() if p.requires_grad)
print(f'BiLSTM Harmonizer parameters: {total_params:,}')

In [ ]:
def train_harm_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    total_tokens = 0
    for sop, tgt, lengths in loader:
        sop, tgt = sop.to(device), tgt.to(device)
        optimizer.zero_grad()
        alto_logits, tenor_logits, bass_logits = model(sop, lengths)

        alto_tgt, tenor_tgt, bass_tgt = tgt[:, :, 0], tgt[:, :, 1], tgt[:, :, 2]
        loss = (criterion(alto_logits.reshape(-1, alto_logits.size(-1)), alto_tgt.reshape(-1)) +
                criterion(tenor_logits.reshape(-1, tenor_logits.size(-1)), tenor_tgt.reshape(-1)) +
                criterion(bass_logits.reshape(-1, bass_logits.size(-1)), bass_tgt.reshape(-1))) / 3.0

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        mask = alto_tgt != PAD_TOKEN
        total_loss += loss.item()
        total_tokens += mask.sum().item()
    return math.exp(total_loss / total_tokens)

@torch.no_grad()
def eval_harm(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    for sop, tgt, lengths in loader:
        sop, tgt = sop.to(device), tgt.to(device)
        alto_logits, tenor_logits, bass_logits = model(sop, lengths)

        alto_tgt, tenor_tgt, bass_tgt = tgt[:, :, 0], tgt[:, :, 1], tgt[:, :, 2]
        loss = (criterion(alto_logits.reshape(-1, alto_logits.size(-1)), alto_tgt.reshape(-1)) +
                criterion(tenor_logits.reshape(-1, tenor_logits.size(-1)), tenor_tgt.reshape(-1)) +
                criterion(bass_logits.reshape(-1, bass_logits.size(-1)), bass_tgt.reshape(-1))) / 3.0

        mask = alto_tgt != PAD_TOKEN
        total_loss += loss.item()
        total_tokens += mask.sum().item()
    return math.exp(total_loss / total_tokens)

In [ ]:
harm_criterion = nn.CrossEntropyLoss(ignore_index=PAD_TOKEN, reduction='sum')
harm_optimizer = torch.optim.Adam(harm_model.parameters(), lr=1e-3)
harm_scheduler = CosineAnnealingLR(harm_optimizer, T_max=NUM_EPOCHS, eta_min=1e-5)

harm_train_ppls, harm_valid_ppls = [], []
best_harm_ppl = float('inf')

for epoch in range(1, NUM_EPOCHS + 1):
    train_ppl = train_harm_epoch(harm_model, train_harm, harm_optimizer, harm_criterion)
    valid_ppl = eval_harm(harm_model, valid_harm, harm_criterion)
    harm_scheduler.step()
    harm_train_ppls.append(train_ppl)
    harm_valid_ppls.append(valid_ppl)

    if valid_ppl < best_harm_ppl:
        best_harm_ppl = valid_ppl
        torch.save(harm_model.state_dict(), 'best_harm.pt')

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d} | Train PPL: {train_ppl:.2f} | Valid PPL: {valid_ppl:.2f}')

harm_model.load_state_dict(torch.load('best_harm.pt', weights_only=True))
print(f'\nBest validation perplexity: {best_harm_ppl:.2f}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, NUM_EPOCHS + 1), harm_train_ppls, label='Train', linewidth=1.5)
ax.plot(range(1, NUM_EPOCHS + 1), harm_valid_ppls, label='Validation', linewidth=1.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('Perplexity')
ax.set_title('Task 2: BiLSTM Harmonizer — Learning Curves')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Evaluation

In [ ]:
harm_test_ppl = eval_harm(harm_model, test_harm, harm_criterion)

print(f'Test Perplexity Comparison:')
print(f'  Frequency baseline: {freq_test_ppl:.2f}')
print(f'  BiLSTM:             {harm_test_ppl:.2f}')

In [ ]:
# Per-voice accuracy
@torch.no_grad()
def compute_accuracy(model, loader):
    model.eval()
    correct = [0, 0, 0]
    pc_correct = [0, 0, 0]
    total = 0
    for sop, tgt, lengths in loader:
        sop, tgt = sop.to(device), tgt.to(device)
        alto_logits, tenor_logits, bass_logits = model(sop, lengths)
        preds = [alto_logits.argmax(-1), tenor_logits.argmax(-1), bass_logits.argmax(-1)]
        targets = [tgt[:, :, 0], tgt[:, :, 1], tgt[:, :, 2]]

        mask = targets[0] != PAD_TOKEN
        total += mask.sum().item()

        for v in range(3):
            correct[v] += ((preds[v] == targets[v]) & mask).sum().item()
            # Pitch-class accuracy: convert tokens to pitches, compare mod 12
            pred_np = preds[v][mask].cpu().numpy()
            tgt_np = targets[v][mask].cpu().numpy()
            for p, t in zip(pred_np, tgt_np):
                pp = idx_to_pitch.get(int(p))
                tp = idx_to_pitch.get(int(t))
                if pp is not None and tp is not None and isinstance(pp, int) and isinstance(tp, int):
                    if pp % 12 == tp % 12:
                        pc_correct[v] += 1

    voice_names = ['Alto', 'Tenor', 'Bass']
    print(f'{"Voice":<8} {"Exact Accuracy":<18} {"Pitch-Class Accuracy":<22}')
    print('-' * 48)
    for v in range(3):
        print(f'{voice_names[v]:<8} {correct[v]/total:>14.1%}     {pc_correct[v]/total:>14.1%}')
    print(f'{"Overall":<8} {sum(correct)/(3*total):>14.1%}     {sum(pc_correct)/(3*total):>14.1%}')

compute_accuracy(harm_model, test_harm)

In [ ]:
# Voice leading analysis: parallel fifths and octaves
def count_parallels(chords):
    """Count parallel fifths and octaves between consecutive chords."""
    p5, p8 = 0, 0
    for t in range(len(chords) - 1):
        curr, nxt = chords[t], chords[t + 1]
        for i in range(4):
            for j in range(i + 1, 4):
                interval_curr = abs(curr[i] - curr[j]) % 12
                interval_next = abs(nxt[i] - nxt[j]) % 12
                motion_i = nxt[i] - curr[i]
                motion_j = nxt[j] - curr[j]
                if motion_i == 0 or motion_j == 0:
                    continue
                if (motion_i > 0) != (motion_j > 0):
                    continue
                if interval_curr == 7 and interval_next == 7:
                    p5 += 1
                if interval_curr == 0 and interval_next == 0:
                    p8 += 1
    return p5, p8

In [ ]:
# Generate harmonizations for test chorales
@torch.no_grad()
def harmonize_chorale(model, soprano_tokens):
    model.eval()
    src = torch.tensor([soprano_tokens], device=device)
    lengths = torch.tensor([len(soprano_tokens)])
    alto_logits, tenor_logits, bass_logits = model(src, lengths)
    return (alto_logits.argmax(-1)[0].cpu().tolist(),
            tenor_logits.argmax(-1)[0].cpu().tolist(),
            bass_logits.argmax(-1)[0].cpu().tolist())

# Compare voice leading across models
num_eval = min(20, len(test_chorales))
results_vl = {'Original Bach': [0, 0], 'Frequency Baseline': [0, 0], 'BiLSTM': [0, 0]}

generated_harms = []
for seq in test_chorales[:num_eval]:
    # Original
    p5, p8 = count_parallels(seq)
    results_vl['Original Bach'][0] += p5
    results_vl['Original Bach'][1] += p8

    # Frequency baseline
    sop = [c[0] for c in seq]
    pred_atb = freq_baseline.predict(sop)
    baseline_chords = [(s, a, t, b) for s, (a, t, b) in zip(sop, pred_atb)]
    p5, p8 = count_parallels(baseline_chords)
    results_vl['Frequency Baseline'][0] += p5
    results_vl['Frequency Baseline'][1] += p8

    # BiLSTM
    sop_tokens = [encode_chord(c)[0] for c in seq]
    alto_pred, tenor_pred, bass_pred = harmonize_chorale(harm_model, sop_tokens)
    lstm_chords = []
    for k in range(len(seq)):
        s = seq[k][0]
        a = idx_to_pitch.get(alto_pred[k], 60)
        t = idx_to_pitch.get(tenor_pred[k], 55)
        b = idx_to_pitch.get(bass_pred[k], 48)
        if all(isinstance(p, int) for p in [s, a, t, b]):
            lstm_chords.append((s, a, t, b))
    generated_harms.append(lstm_chords)
    p5, p8 = count_parallels(lstm_chords)
    results_vl['BiLSTM'][0] += p5
    results_vl['BiLSTM'][1] += p8

print(f'{"Source":<22} {"Parallel 5ths":<16} {"Parallel 8ves":<16}')
print('-' * 54)
for src, (p5, p8) in results_vl.items():
    print(f'{src:<22} {p5:<16} {p8:<16}')

In [ ]:
# Voice range violations for harmonization
print('Voice range violations in harmonized output:')
v_total, t_total = 0, 0
for chords in generated_harms:
    vi, ti = count_range_violations(chords)
    v_total += vi; t_total += ti
print(f'  BiLSTM: {v_total}/{t_total} ({100*v_total/max(t_total,1):.1f}%)')

bv, bt = 0, 0
for seq in test_chorales[:num_eval]:
    sop = [c[0] for c in seq]
    pred_atb = freq_baseline.predict(sop)
    baseline_chords = [(s, a, t, b) for s, (a, t, b) in zip(sop, pred_atb)]
    vi, ti = count_range_violations(baseline_chords)
    bv += vi; bt += ti
print(f'  Baseline: {bv}/{bt} ({100*bv/max(bt,1):.1f}%)')

The BiLSTM harmonizer significantly outperforms the frequency baseline in perplexity, and the per-voice accuracy shows that it learns meaningful harmonic patterns from the soprano context. The voice leading analysis reveals whether the model learns to avoid forbidden parallel motion — a key principle of counterpoint.

### MIDI Export

In [ ]:
# Export the best harmonization
best_harm = max(generated_harms, key=len)
chorales_to_midi(best_harm, 'symbolic_conditioned.mid')
print(f'Exported symbolic_conditioned.mid ({len(best_harm)} chords)')

---

## 4. Results Summary

In [ ]:
results = pd.DataFrame({
    'Task': ['Unconditioned', 'Unconditioned', 'Harmonization', 'Harmonization'],
    'Model': ['Bigram Markov', 'LSTM', 'Frequency Baseline', 'BiLSTM'],
    'Test Perplexity': [markov_test_ppl, lstm_test_ppl, freq_test_ppl, harm_test_ppl],
})
display(results.style.format({'Test Perplexity': '{:.2f}'}).set_caption('Model Comparison'))

### Discussion

**What worked:**
- Data augmentation via transposition was critical for both tasks — it expanded the effective training set by 12x and significantly reduced overfitting
- The LSTM language model captures harmonic relationships within chords through the interleaved representation, and melodic patterns across timesteps
- The bidirectional LSTM harmonizer benefits from seeing the full soprano context — knowing where the melody is headed helps produce more coherent harmonizations
- Cosine annealing learning rate scheduling helped both models converge to lower perplexity

**What didn't:**
- Voice range violations in generated output suggest the model doesn't perfectly learn voice-specific constraints — a constrained decoding approach could help
- The unconditioned model sometimes produces harmonically plausible but structurally aimless sequences — it lacks awareness of musical form (phrases, cadences, sections)

**What we'd try with more time:**
- A Transformer model with positional encodings that are aware of the 4-voice interleaving pattern
- Autoregressive voice-by-voice decoding for harmonization (predict alto conditioned on soprano, then tenor conditioned on soprano+alto, etc.)
- Music-theoretic evaluation using chord label analysis and key detection
- Beam search or constrained sampling to enforce voice range and parallel motion rules

---

## 5. Related Work

### Unconditioned Symbolic Generation

The JSB Chorales dataset was introduced as a benchmark for music modeling by Boulanger-Lewandowski, Bengio, and Vincent (2012), who applied RNN-RBM and RTRBM models to polyphonic music generation. Their work established the convention of evaluating models on this dataset using log-likelihood — a standard we follow with perplexity. They reported negative log-likelihood scores in the range of 7-8 nats per timestep on JSB Chorales, setting a baseline that subsequent architectures aimed to beat.

LSTM-based approaches for music generation gained traction through several key works. Eck and Schmidhuber (2002) were among the first to apply LSTMs to music composition, demonstrating that the gating mechanism allows the network to learn long-range temporal structure like chord progressions. Sturm et al. (2016) later trained character-level LSTMs on folk music in ABC notation, showing that recurrent models can capture musical structure even from raw text-based symbolic representations. The survey by Briot, Hadjeres, and Pachet (2020), published in *Neural Computing and Applications*, provides a comprehensive taxonomy of deep learning methods for music generation, categorizing approaches by representation (symbolic vs. audio), architecture (RNN, CNN, GAN, VAE, Transformer), and generation strategy (single-step, iterative, reinforcement learning).

Our LSTM language model follows the same general paradigm as Boulanger-Lewandowski et al. but uses an interleaved SATB token representation rather than their piano-roll encoding. This trades spatial structure for sequential simplicity, allowing us to apply standard language modeling techniques directly.

**References:**
- Boulanger-Lewandowski, N., Bengio, Y., & Vincent, P. (2012). Modeling temporal dependencies in high-dimensional sequences: Application to polyphonic music generation and transcription. *ICML 2012*.
- Eck, D., & Schmidhuber, J. (2002). A first look at music composition using LSTM recurrent neural networks. *IDSIA Technical Report*.
- Sturm, B. L., Santos, J. F., Ben-Tal, O., & Korshunova, I. (2016). Music transcription modelling and composition using deep learning. *Proc. 1st Conference on Computer Simulation of Musical Creativity*.
- Briot, J.-P., Hadjeres, G., & Pachet, F.-D. (2020). Deep learning techniques for music generation — A survey. *Neural Computing and Applications*, 32(4), 981–993.

### Conditioned Generation (Harmonization)

Chorale harmonization has a long history in computational musicology. Ebcioglu (1988) developed an expert system with over 350 hand-coded rules for Bach-style harmonization, implementing constraints from music theory textbooks. The system produced results that music theorists found acceptable but was brittle — adding new rules could break existing ones, highlighting the limitations of rule-based approaches.

The transition to data-driven methods began with Hild, Feulner, and Menzel (1992), whose HARMONET system used neural networks for chorale harmonization, training on 20 Bach chorales and producing reasonable results despite the small dataset. Allan and Williams (2005) later applied Hidden Markov Models to harmonization on a dataset of Bach chorales, using multiple coupled HMM chains to model the four voices.

The most influential recent work is DeepBach by Hadjeres, Pachet, and Nielsen (2017), which models harmonization using a Gibbs sampling framework with deep neural networks. DeepBach generates each voice iteratively, conditioned on the other voices and metadata (fermatas, key signatures), producing harmonizations that experts rated as comparable to Bach in blind listening tests — 50% of evaluators could not distinguish DeepBach from real Bach. Huang, Cooijmans, Roberts, Courville, and Eck (2017) also explored attention-based models for music generation, finding that self-attention mechanisms help capture long-range harmonic dependencies.

Our bidirectional LSTM approach is simpler than DeepBach — we predict all three lower voices simultaneously rather than iteratively — but shares the key insight that context in both temporal directions is critical for good harmonization. The BiLSTM architecture naturally provides this through its forward and backward passes over the soprano melody.

**References:**
- Ebcioglu, K. (1988). An expert system for harmonizing four-part chorales. *Computer Music Journal*, 12(3), 43–51.
- Hild, H., Feulner, J., & Menzel, W. (1992). HARMONET: A neural net for harmonizing chorales in the style of J.S. Bach. *Advances in Neural Information Processing Systems (NeurIPS) 4*.
- Allan, M., & Williams, C. K. I. (2005). Harmonising chorales by probabilistic inference. *Advances in Neural Information Processing Systems (NeurIPS) 17*.
- Hadjeres, G., Pachet, F., & Nielsen, F. (2017). DeepBach: A steerable model for Bach chorales generation. *ICML 2017*.
- Huang, C.-Z. A., Cooijmans, T., Roberts, A., Courville, A., & Eck, D. (2017). Counterpoint by convolution. *ISMIR 2017*.